In [2]:
import pandas as pd
from scipy import signal
import math
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
import pandas as pd
import numpy as np
import math


def import_data(file_path):
    """Import data from a *.glm file and store it in a dataframe"""

    # Colonnes réellement nécessaires
    ColNames = [
        'time (s)',

        'Fxal (N)', 'Fyal (N)', 'Fzal (N)',
        'Txal (Nm)', 'Tyal (Nm)', 'Tzal (Nm)',

        'Fxar (N)', 'Fyar (N)', 'Fzar (N)',
        'Txar (Nm)', 'Tyar (Nm)', 'Tzar (Nm)',

        'GF (N)', 'LFv (N)', 'LFh (N)', 'LFt (N)'
    ]

    # Noms simplifiés
    NewColNames = [
        'time',

        'Fxal','Fyal','Fzal',
        'Txal','Tyal','Tzal',

        'Fxar','Fyar','Fzar',
        'Txar','Tyar','Tzar',

        'GF','LFv','LFh','LFt'
    ]

    df = pd.read_csv(
        file_path,
        sep="\t",
        header=0,
        usecols=ColNames,
        engine="python"
    )

    df.columns = NewColNames

    return df


def compute_cop(glm_df, baseline, z0=0.00155, angle=0.523599):
    """Compute center of pressure (CPL, CPR)"""

    # Forces ATI frame
    Fal = -np.array([
        glm_df['Fxal'],
        glm_df['Fyal'],
        glm_df['Fzal']
    ])

    Far = -np.array([
        glm_df['Fxar'],
        glm_df['Fyar'],
        glm_df['Fzar']
    ])

    Tal = -np.array([
        glm_df['Txal'],
        glm_df['Tyal'],
        glm_df['Tzal']
    ])

    Tar = -np.array([
        glm_df['Txar'],
        glm_df['Tyar'],
        glm_df['Tzar']
    ])

    # Baseline correction
    Fal = Fal - np.nanmean(Fal[:, baseline], axis=1).reshape((3,1))
    Far = Far - np.nanmean(Far[:, baseline], axis=1).reshape((3,1))

    Tal = Tal - np.nanmean(Tal[:, baseline], axis=1).reshape((3,1))
    Tar = Tar - np.nanmean(Tar[:, baseline], axis=1).reshape((3,1))

    # CoP calculation
    CPL_ati = -np.array([
        (Tal[1,:] + Fal[0,:]*z0) / Fal[2,:],
        -(Tal[0,:] - Fal[1,:]*z0) / Fal[2,:]
    ])

    CPR_ati = -np.array([
        (Tar[1,:] + Far[0,:]*z0) / Far[2,:],
        -(Tar[0,:] - Far[1,:]*z0) / Far[2,:]
    ])

    # Conversion repère manipulandum
    CPL = -CPL_ati[0,:]*math.sin(angle) - CPL_ati[1,:]*math.cos(angle)
    CPR =  CPR_ati[0,:]*math.sin(angle) + CPR_ati[1,:]*math.cos(angle)

    return CPL, CPR

In [5]:
# dossier contenant les fichiers
dossier = "Groupe4_LUNDI_APREM"

# liste des fichiers que tu veux traiter
fichiers_S01 = [
    "Gr4_S01_L1_001.glm",
    "Gr4_S01_L1_002.glm",
    "Gr4_S01_L1_003.glm",
    "Gr4_S01_L1_004.glm",
    "Gr4_S01_L2_001.glm",
    "Gr4_S01_L2_002.glm",
    "Gr4_S01_L2_003.glm",
    "Gr4_S01_L2_004.glm",
    "Gr4_S01_L3_001.glm",
    "Gr4_S01_L3_002.glm",
    "Gr4_S01_L3_003.glm",
    "Gr4_S01_L3_004.glm",
    "Gr4_S01_L4_001.glm",
    "Gr4_S01_L4_002.glm",
    "Gr4_S01_L4_003.glm",
    "Gr4_S01_L4_004.glm",
]

# dossier où sauvegarder les figures
output_folder = "grip_S01"
os.makedirs(output_folder, exist_ok=True)

# baseline pour la correction
baseline = np.arange(0, 200)

# fenêtre temporelle à afficher
tmin = 0   # secondes
tmax = 43   # secondes

# boucle sur les fichiers
for file_name in fichiers_S01:
    file_path = os.path.join(dossier, file_name)
    
    if not os.path.exists(file_path):
        print(f"⚠️ Fichier non trouvé : {file_name}")
        continue
    
    # importer les données
    df = import_data(file_path)
    
    # calcul du centre de pression
    CPL, CPR = compute_cop(df, baseline)
    
    # récupérer le temps
    time = df["time"].values
    
    # -----------------------------
    # filtrage temporel
    # -----------------------------
    mask = (time >= tmin) & (time <= tmax)

    time_plot = time[mask]
    Fzal_plot = df["Fzal"].values[mask]
    Fzar_plot = df["Fzar"].values[mask]
    CPL_plot = CPL[mask]
    CPR_plot = CPR[mask]
    GF_plot = df["GF"].values[mask]

    # -----------------------------
    # création de la figure
    # -----------------------------
    plt.figure(figsize=(12,8))

    # forces verticales
    plt.subplot(3,1,1)
    plt.plot(time_plot, Fzal_plot, label="Fz left")
    plt.plot(time_plot, Fzar_plot, label="Fz right")
    plt.title(f"Vertical forces ({tmin}-{tmax} s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.legend()

    # centre de pression
    plt.subplot(3,1,2)
    plt.plot(time_plot, CPL_plot, label="CPL")
    plt.plot(time_plot, CPR_plot, label="CPR")
    plt.title(f"Center of Pressure ({tmin}-{tmax} s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Position (m)")
    plt.legend()

    # grip force
    plt.subplot(3,1,3)
    plt.plot(time_plot, GF_plot)
    plt.title(f"Grip Force ({tmin}-{tmax} s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")

    plt.tight_layout()

    # -----------------------------
    # générer un nom de fichier simplifié
    # -----------------------------
    parts = file_name.split('_')
    groupe = parts[1]
    ligne = parts[2]
    numero = str(int(parts[3].split('.')[0]))

    simple_name = f"{groupe}_{ligne}_{numero}.png"
    save_path = os.path.join(output_folder, simple_name)

    plt.savefig(save_path)
    plt.close()

    # -----------------------------
    # print moyenne baseline
    # -----------------------------
    print(f"{file_name} → Moyenne 10 premières valeurs CPL : {np.mean(CPL[:10]):.5f}, CPR : {np.mean(CPR[:10]):.5f}")
    print(f"          Fzal : {np.mean(df['Fzal'][:10]):.5f}, Fzar : {np.mean(df['Fzar'][:10]):.5f}")

Gr4_S01_L1_001.glm → Moyenne 10 premières valeurs CPL : 0.00879, CPR : 0.00417
          Fzal : -0.01941, Fzar : -0.00224
Gr4_S01_L1_002.glm → Moyenne 10 premières valeurs CPL : 0.00003, CPR : -0.00175
          Fzal : -0.02366, Fzar : -0.00268
Gr4_S01_L1_003.glm → Moyenne 10 premières valeurs CPL : 0.00026, CPR : 0.00889
          Fzal : -0.01540, Fzar : -0.00369
Gr4_S01_L1_004.glm → Moyenne 10 premières valeurs CPL : -0.00212, CPR : -0.00461
          Fzal : 0.00083, Fzar : -0.00000
Gr4_S01_L2_001.glm → Moyenne 10 premières valeurs CPL : -0.00211, CPR : -0.03779
          Fzal : -0.02439, Fzar : -0.00354
Gr4_S01_L2_002.glm → Moyenne 10 premières valeurs CPL : -0.02638, CPR : -0.00008
          Fzal : -0.02181, Fzar : -0.00464
Gr4_S01_L2_003.glm → Moyenne 10 premières valeurs CPL : 0.01551, CPR : 0.02459
          Fzal : -0.02589, Fzar : -0.00094
Gr4_S01_L2_004.glm → Moyenne 10 premières valeurs CPL : 0.00114, CPR : 0.00067
          Fzal : 0.00169, Fzar : 0.00024
Gr4_S01_L3_001.glm →

In [12]:
# dossier où sauvegarder les figures moyennes
output_folder_mean = "grip_S01"
os.makedirs(output_folder_mean, exist_ok=True)  # crée le dossier s'il n'existe pas

# dictionnaire pour stocker les données par Lx
data_par_L = {}

for file_name in fichiers_S01:
    file_path = os.path.join(dossier, file_name)
    if not os.path.exists(file_path):
        print(f"⚠️ Fichier non trouvé : {file_name}")
        continue
    
    df = import_data(file_path)
    CPL, CPR = compute_cop(df, baseline)
    
    # récupérer Lx
    ligne = file_name.split('_')[2]  # L1, L2, etc.
    
    # initialiser les listes si pas encore fait
    if ligne not in data_par_L:
        data_par_L[ligne] = {
            "Fzal": [],
            "Fzar": [],
            "CPL": [],
            "CPR": [],
            "GF": []
        }
    
    # ajouter les séries au dictionnaire
    data_par_L[ligne]["Fzal"].append(df["Fzal"].values)
    data_par_L[ligne]["Fzar"].append(df["Fzar"].values)
    data_par_L[ligne]["CPL"].append(CPL)
    data_par_L[ligne]["CPR"].append(CPR)
    data_par_L[ligne]["GF"].append(df["GF"].values)

# -----------------------------
# calculer la moyenne pour chaque Lx et sauvegarder la figure
# -----------------------------
for ligne, d in data_par_L.items():
    
    # trouver la longueur minimale pour cette ligne
    min_len = min([len(arr) for arr in d["Fzal"]])
    
    # tronquer toutes les séries à min_len
    Fzal_stack = np.vstack([arr[:min_len] for arr in d["Fzal"]])
    Fzar_stack = np.vstack([arr[:min_len] for arr in d["Fzar"]])
    CPL_stack  = np.vstack([arr[:min_len] for arr in d["CPL"]])
    CPR_stack  = np.vstack([arr[:min_len] for arr in d["CPR"]])
    GF_stack   = np.vstack([arr[:min_len] for arr in d["GF"]])
    
    # calculer la moyenne
    Fzal_mean = np.mean(Fzal_stack, axis=0)
    Fzar_mean = np.mean(Fzar_stack, axis=0)
    CPL_mean  = np.mean(CPL_stack, axis=0)
    CPR_mean  = np.mean(CPR_stack, axis=0)
    GF_mean   = np.mean(GF_stack, axis=0)
    
    # générer le temps
    time = np.arange(min_len) * 0.00125  # ajuster selon fréquence d'échantillonnage
    
    # -----------------------------
    # création de la figure
    # -----------------------------
    plt.figure(figsize=(12,8))
    
    plt.subplot(3,1,1)
    plt.plot(time, Fzal_mean, label="Fz left")
    plt.plot(time, Fzar_mean, label="Fz right")
    plt.title(f"Vertical forces moyennes {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.legend()
    
    plt.subplot(3,1,2)
    plt.plot(time, CPL_mean, label="CPL")
    plt.plot(time, CPR_mean, label="CPR")
    plt.title(f"Center of Pressure moyennes {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Position (m)")
    plt.legend()
    
    plt.subplot(3,1,3)
    plt.plot(time, GF_mean)
    plt.title(f"Grip Force moyenne {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    
    plt.tight_layout()
    
    # -----------------------------
    # sauvegarde de la figure
    # -----------------------------
    save_name = f"{ligne}_moyenne.png"
    save_path = os.path.join(output_folder_mean, save_name)
    plt.savefig(save_path)
    plt.close()  # ferme la figure pour accélérer
    
    print(f"✅ Figure moyenne sauvegardée pour {ligne} → {save_name}")

✅ Figure moyenne sauvegardée pour L1 → L1_moyenne.png
✅ Figure moyenne sauvegardée pour L2 → L2_moyenne.png
✅ Figure moyenne sauvegardée pour L3 → L3_moyenne.png
✅ Figure moyenne sauvegardée pour L4 → L4_moyenne.png
